In [3]:
# learning rate evaluation
import tensorflow as tf
import numpy as np

x_data = np.array([[1, 2, 1],
                   [1, 3, 2],
                   [1, 3, 4],
                   [1, 5, 5],
                   [1, 7, 5],
                   [1, 2, 5],
                   [1, 6, 6],
                   [1, 7, 7]], dtype=np.float32)

y_data = np.array([[0, 0, 1],
                   [0, 0, 1],
                   [0, 0, 1],
                   [0, 1, 0],
                   [0, 1, 0],
                   [0, 1, 0],
                   [1, 0, 0],
                   [1, 0, 0]], dtype=np.float32)

x_test = np.array([[2, 1, 1],
                   [3, 1, 2],
                   [3, 3, 4]], dtype=np.float32)

y_test = np.array([[0, 0, 1],
                   [0, 0, 1],
                   [0, 0, 1]], dtype=np.float32)

learning_rate = 0.1

# 모델 정의
model = tf.keras.Sequential([
    tf.keras.layers.Dense(units=3, input_dim=3, activation='softmax')
])

model.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              metrics=['accuracy'])

model.fit(x_data, y_data, epochs=1000, verbose=0)

# predict_classes는 deprecated. argmax 사용
pred = model.predict(x_test)
pred_classes = np.argmax(pred, axis=1)
print("Prediction classes:", pred_classes)

# Accuracy
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print("Accuracy:", acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Prediction classes: [2 2 2]
Accuracy: 1.0


In [1]:
# linear regression without min-max scaling

import tensorflow as tf
import numpy as np

xy = np.array([[828.659973, 833.450012, 908100, 828.349976, 831.659973],
               [823.02002, 828.070007, 1828100, 821.655029, 828.070007],
               [819.929993, 824.400024, 1438100, 818.97998, 824.159973],
               [816, 820.958984, 1008100, 815.48999, 819.23999],
               [819.359985, 823, 1188100, 818.469971, 818.97998],
               [819, 823, 1198100, 816, 820.450012],
               [811.700012, 815.25, 1098100, 809.780029, 813.669983],
               [809.51001, 816.659973, 1398100, 804.539978, 809.559998]])

x_data = xy[:, 0:-1]
y_data = xy[:, [-1]]

# 모델 정의
model = tf.keras.Sequential([
    tf.keras.layers.Dense(units=1, input_dim=4, activation='linear')
])

model.compile(loss='mse', optimizer=tf.keras.optimizers.SGD(learning_rate=1e-5))
model.summary()

# 학습
# 문제점: 입력 값 스케일이 너무 크거나 차이가 많이 나면
# 학습이 불안정 (loss = NaN)
# 해결 방법 → Min-Max scaling
history = model.fit(x_data, y_data, epochs=100, verbose=0)
print("Loss (without scaling):", history.history['loss'])



C:\Users\njy60\anaconda3\envs\tf\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 1)                   │               5 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5 (20.00 B)

 Trainable params: 5 (20.00 B)

 Non-trainable params: 0 (0.00 B)

Loss (without scaling): [259692544000.0, 2.8531907410851076e+26, inf, inf, inf, inf, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]


In [2]:
# linear regression with min-max scaling
import tensorflow as tf
import numpy as np

# Min-Max Scaling(데이터 전처리):
    # 모든 feature를 0~1 범위로 변환
    # 학습 안정화 → gradient가 너무 크거나 작지 않게 조정
def min_max_scaler(data):
    numerator = data - np.min(data, axis=0)
    denominator = np.max(data, axis=0) - np.min(data, axis=0)
    return numerator / (denominator + 1e-7)  # divide by zero 방지

xy_scaled = min_max_scaler(xy)
x_data = xy_scaled[:, 0:-1]
y_data = xy_scaled[:, [-1]]

model = tf.keras.Sequential([
    tf.keras.layers.Dense(units=1, input_dim=4, activation='linear')
])

model.compile(loss='mse', optimizer=tf.keras.optimizers.SGD(learning_rate=1e-5))
model.summary()

history = model.fit(x_data, y_data, epochs=1000, verbose=0)

predictions = model.predict(x_data)
score = model.evaluate(x_data, y_data, verbose=0)

print('Prediction (with scaling):\n', predictions)
print('Cost:', score)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                      │ (None, 1)                   │               5 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5 (20.00 B)

 Trainable params: 5 (20.00 B)

 Non-trainable params: 0 (0.00 B)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Prediction (with scaling):
 [[-1.4574764 ]
 [-1.2864645 ]
 [-0.93709254]
 [-0.5429628 ]
 [-0.80240095]
 [-0.75135165]
 [-0.185392  ]
 [-0.15098874]]
Cost: 2.1604738235473633
